In [ ]:
📦 Google Drive already mounted
📦 Repository already exists — skipping clone
🚀 Running setup_fenicsx.py in current kernel
======================================================================
🔧 FEniCSx Setup Configuration
======================================================================
PETSc type      : real
Clean install   : True
======================================================================

📦 Google Drive detected — using persistent cache

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

======================================================================
✅ FEniCSx setup complete!
======================================================================

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex
======================================================================

🐍 Python          : 3.11.15
📦 dolfinx         : 0.10.0
💻 Platform        : Linux-6.6.113+-x86_64-with-glibc2.35
🧵 Running as root : True

🔎 fenicsx runtime info
-----------------------
Environment        : fenicsx
micromamba         : /content/micromamba/bin/micromamba
MPI implementation : OPENMPI
MPI version        : mpiexec (OpenRTE) 4.1.2
MPI ranks (-np)    : 4

In [ ]:
%%fenicsx -np 4

from pathlib import Path

from mpi4py import MPI
import gmsh
from dolfinx.io import XDMFFile

# DOLFINx version-compatible import
try:
    from dolfinx.io import gmsh as gmshio
except ImportError:
    try:
        import dolfinx.io.gmshio as gmshio
    except ImportError:
        import dolfinx.io.gmsh as gmshio


def generate_disk(filename: Path, res: float, order: int = 1, refinement_level: int = 0):
    """Generate a unit disk mesh centered at the origin and save as XDMF."""
    comm = MPI.COMM_WORLD
    model_rank = 0
    gdim = 2

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 1)

    try:
        if comm.rank == model_rank:
            gmsh.model.add("disk")
            gmsh.model.setCurrent("disk")

            disk = gmsh.model.occ.addDisk(0.0, 0.0, 0.0, 1.0, 1.0)
            gmsh.model.occ.synchronize()

            gmsh.model.addPhysicalGroup(gdim, [disk], tag=1)
            gmsh.model.setPhysicalName(gdim, 1, "Disk")

            gmsh.option.setNumber("Mesh.CharacteristicLengthMin", res)
            gmsh.option.setNumber("Mesh.CharacteristicLengthMax", res)

            gmsh.model.mesh.generate(gdim)
            gmsh.model.mesh.setOrder(order)

            for _ in range(refinement_level):
                gmsh.model.mesh.refine()
                gmsh.model.mesh.setOrder(order)

        mesh_data = gmshio.model_to_mesh(gmsh.model, comm, model_rank, gdim=gdim)

        # Support both old/new return formats
        if hasattr(mesh_data, "mesh"):
            msh = mesh_data.mesh
        else:
            msh = mesh_data[0]

        # Important: keep the mesh name fixed for read_mesh(name="mesh")
        msh.name = "mesh"

    finally:
        gmsh.finalize()

    out_name = filename.with_stem(f"{filename.stem}_{refinement_level}").with_suffix(".xdmf")
    out_name.parent.mkdir(parents=True, exist_ok=True)

    with XDMFFile(comm, str(out_name), "w") as xdmf:
        xdmf.write_mesh(msh)

    if comm.rank == 0:
        print(f"Saved mesh to {out_name}")


# Generate meshes with refinement levels 0,1,2,3
for i in range(4):
    generate_disk(Path("meshes/disk.xdmf"), res=0.1, order=2, refinement_level=i)